<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 6 — Fine-tuning de BERT</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Un backbone, tres cabezas — con datasets reales y demo sobre su corpus</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Entorno: Google Colab con GPU T4 (free tier).** Entorno de ejecución → Cambiar tipo → **GPU**. Los tres modelos (BETO 110M, Sentence-BERT) caben de sobra en los 15 GB de la T4 usando `fp16`. Esta versión entrena con **datasets reales de HuggingFace** (no con el corpus de juguete): `tweet_sentiment_multilingual` para clasificación y `conll2002` para NER. El corpus chiapaneco se usa solo para la **inferencia final** de cada parte. Las celdas de *liberar memoria* permiten correr A → B → C en una sola sesión sin saturar la VRAM.


## Objetivo

Tres partes, el mismo BERT preentrenado en español como base. **A)** Afinar un encoder de oraciones (Sentence-BERT) con sus pares del Lab 3. **B)** Clasificar **sentimiento** con un dataset real en español. **C)** **NER** con CoNLL-2002. Cada parte cierra **usando el modelo afinado** sobre el corpus chiapaneco, y libera memoria antes de la siguiente.


## 0 · Setup, GPU y utilidades

In [ ]:
# Instalación (Colab). Reiniciar el entorno si lo pide tras instalar.
!pip install -q transformers sentence-transformers datasets seqeval accelerate

import gc, math, json
import numpy as np
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — activen el runtime de GPU')

def liberar_memoria():
    """Libera RAM/VRAM. Llamar tras borrar (del) las variables del entrenamiento previo."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f'VRAM en uso: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# El corpus chiapaneco (del Lab 1) se usa SOLO para la inferencia final de cada parte.
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}
ids = [d['id'] for d in corpus]; titulos = {d['id']: d['titulo'] for d in corpus}
print(len(corpus), 'documentos del corpus cargados (para inferencia).')

---
## Parte A · Embeddings con Sentence-BERT (datos: sus qrels del Lab 3)

> **Por qué aquí sí usamos el corpus.** El fine-tuning de embeddings necesita pares *de dominio* (consulta ↔ documento relevante). Esos pares salen de **sus** qrels del Lab 3: es el cierre del arco de la unidad, donde su juicio de relevancia se vuelve señal de entrenamiento. **Advertencia metodológica:** con ~5 consultas esto es una *demostración del método*, no un experimento — la mejora puede ser chica o ruidosa. Lo evaluable es el pipeline correcto, no el número.


**A.1** Línea base: carguen un Sentence-BERT en español y midan su buscador **sin afinar** con su nDCG del Lab 3.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
modelo = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')
# TODO: reusen sus qrels del Lab 3; funciones emb_corpus, buscar (coseno) y ndcg_medio.
# TODO: midan el nDCG del modelo SIN afinar (linea base a superar).
raise NotImplementedError

**A.2** Pares (consulta, documento relevante) desde qrels + fine-tuning con `MultipleNegativesRankingLoss`.

In [ ]:
# TODO: InputExample(texts=[consulta, documento]) para positivos (g>=2); DataLoader;
#       MultipleNegativesRankingLoss; modelo.fit(epochs=2, warmup_steps=...). Recalcular nDCG.
raise NotImplementedError

_Reporten los tres nDCG (FastText, SBERT base, SBERT afinado) y comenten:_ 

**A.3 · Uso del modelo afinado.** Busquen con una consulta nueva usando el encoder ya entrenado.

In [ ]:
# TODO: con el modelo afinado y EMB1, busquen una consulta nueva de su eleccion (k=5).
raise NotImplementedError

**Liberar memoria** antes del siguiente entrenamiento (clave en T4).

In [ ]:
# Borren las variables del modelo de esta parte y liberen VRAM:
del modelo
liberar_memoria()

---
## Parte B · Clasificación de sentimiento (dataset real en español)

Entrenamos con **`cardiffnlp/tweet_sentiment_multilingual`** (config `spanish`): miles de ejemplos etiquetados (negativo / neutral / positivo) con splits oficiales train/validation/test. *Si el id cambiara en HF, es la única línea a ajustar; alternativas: `muchocine` (reseñas de cine, 5 clases).*


**B.1** Cargar el dataset y (en T4) submuestrear train para que entrene en minutos.

In [ ]:
from datasets import load_dataset
ds = load_dataset('cardiffnlp/tweet_sentiment_multilingual', 'spanish')
# TODO: obtener la lista de clases (features['label'].names) y los mapeos id2lab/lab2id.
# TODO: submuestrear ds['train'] (~2000) para T4 y tomar ds['test'].
raise NotImplementedError

**B.2** Tokenizar con el tokenizer de BETO.

In [ ]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)
# TODO: tokenizar (truncation + padding, max_length=128) train y test -> ds_tr, ds_te.
raise NotImplementedError

**B.3** Fine-tuning con `Trainer` (LR 2e-5, pocas épocas, `fp16` para la T4).

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
# TODO: AutoModelForSequenceClassification (num_labels, id2label, label2id);
#       compute_metrics (accuracy + f1_macro); TrainingArguments (lr 2e-5, fp16=True en T4);
#       Trainer; entrenar y evaluar.
raise NotImplementedError

**B.4** Análisis de errores: matriz de confusión y reporte por clase.

In [ ]:
# TODO: classification_report + confusion_matrix sobre el test. ¿Que clase se confunde mas?
raise NotImplementedError

_¿Qué clase es la más difícil? ¿Accuracy o F1-macro es mejor criterio aquí? ¿Por qué?_ 

**B.5 · Uso del modelo afinado** — transferencia al corpus chiapaneco. El modelo se entrenó con tuits; veamos qué predice sobre noticias.

In [ ]:
from transformers import pipeline
# TODO: crear un pipeline('text-classification') con su modelo afinado y aplicarlo a
#       3-4 frases propias + algunas del corpus (crudo[...]). Comenten el domain shift.
raise NotImplementedError

**Liberar memoria** antes de la Parte C.

In [ ]:
del model            # y el trainer/pipeline si los nombraron distinto
liberar_memoria()

---
## Parte C · NER con CoNLL-2002 (español)

Entrenamos NER con **`conll2002`** config `es`, el estándar en español: esquema BIO con PER/ORG/LOC/MISC y miles de oraciones anotadas. *Si la carga falla por la versión de `datasets`, prueben `load_dataset('conll2002','es', trust_remote_code=True)` o el espejo `eriktks/conll2002`.*


**C.1** Cargar el dataset y leer el esquema de etiquetas desde sus features.

In [ ]:
from datasets import load_dataset
conll = load_dataset('conll2002', 'es', trust_remote_code=True)
# TODO: obtener 'etiquetas' (features['ner_tags'].feature.names) y los mapeos id2lab/lab2id.
raise NotImplementedError

**C.2 — el corazón del lab.** Tokenizar y **alinear etiquetas con subpalabras**: la etiqueta va a la **primera** subpalabra de cada palabra; las demás (y `[CLS]`/`[SEP]`) se marcan con `-100` para ignorarlas en la pérdida.

In [ ]:
from transformers import AutoTokenizer
CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)
def tokeniza_y_alinea(batch):
    # TODO: tok(..., is_split_into_words=True); por cada ejemplo usar enc.word_ids(batch_index=i)
    #       para poner la etiqueta SOLO en la primera subpalabra y -100 en las demas y en [CLS]/[SEP].
    raise NotImplementedError
# conll_tok = conll.map(tokeniza_y_alinea, batched=True, remove_columns=conll['train'].column_names)

**C.3** Fine-tuning con `AutoModelForTokenClassification` (`fp16`, T4).

In [ ]:
from transformers import (AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification)
# TODO: AutoModelForTokenClassification (num_labels, id2label, label2id);
#       DataCollatorForTokenClassification; TrainingArguments (fp16 en T4); Trainer; entrenar.
raise NotImplementedError

**C.4** Evaluación con **seqeval** (a nivel de entidad, no de token).

In [ ]:
from seqeval.metrics import classification_report as seq_report
# TODO: predecir; reconstruir secuencias ignorando -100; evaluar con seqeval (entidad).
#       Expliquen por que NO se usa accuracy por token.
raise NotImplementedError

_¿Por qué seqeval y no accuracy por token? ¿Qué tipo de entidad cuesta más?_ 

**C.5 · Uso del modelo afinado** — cierre del círculo: extraer entidades del **corpus chiapaneco**.

In [ ]:
from transformers import pipeline
# TODO: pipeline('ner', aggregation_strategy='simple') con su modelo; aplicarlo a 3 documentos
#       del corpus (crudo[...]) e imprimir las entidades detectadas con su tipo y score.
raise NotImplementedError

**Liberar memoria** al terminar.

In [ ]:
del model
liberar_memoria()

---
## Síntesis

Las tres partes usaron **el mismo BERT preentrenado** y solo cambiaron la cabeza y los datos: cabeza siamesa con sus qrels (A), `[CLS]` + lineal con un dataset de sentimiento real (B), y una etiqueta por token con CoNLL-2002 (C). Cada modelo afinado se aplicó **sobre su propio corpus**, y entre entrenamientos se liberó memoria para sostener la sesión en una T4. Ese paradigma —preentrenar una vez, adaptar barato— es la base de los sistemas RAG de la Unidad 3.


## Entregables — Lab 6
- [ ] **A:** fine-tuning de Sentence-BERT + los tres nDCG + búsqueda con el modelo afinado.
- [ ] **B:** clasificación con dataset real + matriz de confusión + inferencia sobre el corpus.
- [ ] **C:** NER con CoNLL-2002 + alineación de subpalabras + seqeval + extracción sobre el corpus.
- [ ] Celdas de liberación de memoria ejecutadas entre partes.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
